In [1]:
from utils import file
import matplotlib.pyplot as plt 
import segmentation_models as sm
import tensorflow as tf
import tensorflow_io as tfio
from models import attention_unet

Segmentation Models: using `keras` framework.


In [2]:
TRAINING_IMAGE_PATH = 'training/images'
TRAINING_MASK_PATH = 'training/masks'

trainingImagePaths = file.getSortedFilePaths(TRAINING_IMAGE_PATH)
trainingMaskPaths = file.getSortedFilePaths(TRAINING_MASK_PATH)

trainDataset = file.datasetGenerator(trainingImagePaths, trainingMaskPaths)
trainDataset = trainDataset.prefetch(buffer_size=tf.data.AUTOTUNE)

def channels_first(image, mask):
    image = tf.transpose(image, perm=[0, 3, 1, 2])
    mask = tf.transpose(mask, perm=[0, 3, 1, 2])
    return image, mask

trainDataset = trainDataset.map(channels_first)

trainDataset

<MapDataset element_spec=(TensorSpec(shape=(None, 3, 256, 256), dtype=tf.int32, name=None), TensorSpec(shape=(None, 1, 256, 256), dtype=tf.int32, name=None))>

In [7]:
current_loss = "dice"

class DisplayCallback(tf.keras.callbacks.Callback):
    def __init__(self, dataset, model, epoch_interval=1):
        self.dataset = dataset
        self.epoch_interval = epoch_interval
        self.best_val_score = 0.0
        self.model = model
    
    def display(self, display_list, extra_title=''):
        plt.figure(figsize=(15, 15))
        title = ['Tikroji nuotrauka', 'Tikroji segmentacija', 'Segmentuota nuotrauka']

        if len(display_list) > len(title):
            title.append(extra_title)

        for i in range(len(display_list)):
            plt.subplot(1, len(display_list), i+1)
            plt.title(title[i])
            plt.imshow(display_list[i], cmap='gray')
            plt.axis('off')
        plt.show()
        
    def create_mask(self, pred_mask):
        pred_mask = (pred_mask > 0.5).astype("int32")
        return pred_mask[0]
    
    def show_predictions(self, dataset, num=2):
        for image, mask in dataset.take(num):
            pred_mask = self.model.predict(image)
            self.display([image[0], mask[0], self.create_mask(pred_mask)])
        
    def on_epoch_end(self, epoch, logs=None):
        # also save if validation error is smallest
        print(logs.keys())
        if 'get_f1' in logs.keys():
            val_score = logs['get_f1']
            if val_score > self.best_val_score:
                self.best_val_score = val_score
                print('New best weights found!')
                self.model.save_weights("./attention-unet-weights/" + current_loss + '/best_weights.hdf5')
        else:
            print('Key val_dice_eval does not exist!')
            
        # if epoch and epoch % self.epoch_interval == 0:
        #     self.show_predictions(self.dataset)
        #     print ('\nSample Prediction after epoch {}\n'.format(epoch+1))

from keras.callbacks import ModelCheckpoint


mcp_save = ModelCheckpoint(
    './attention-unet-weights/' +  current_loss + '/weights.{epoch:02d}-{loss:.4f}.hdf5', save_best_only=False, save_weights_only=True, monitor='loss', verbose=1)

In [24]:
import tensorflow.keras.backend as K

def get_f1(y_true, y_pred):
    y_true = K.squeeze(y_true, axis=1) # remove second channel if present
    y_pred = K.squeeze(y_pred, axis=1)
    if K.image_data_format() == 'channels_first': # transpose axes to channels last
        y_true = K.permute_dimensions(y_true, [0, 2, 3, 1])
        y_pred = K.permute_dimensions(y_pred, [0, 2, 3, 1])
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    f1_val = 2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val


# def dice_loss(y_true, y_pred):
#     return 1 - dice_coef(y_true, y_pred)

In [25]:
# define optimizer
optimizer = tf.keras.optimizers.Adam(0.0001)
bce   = tf.keras.losses.BinaryCrossentropy()

def dice_loss(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = tf.reduce_mean((2. * intersection + smooth) / (union + smooth), axis=0)
    return 1 - dice

def focal_loss(y_true, y_pred):
    gamma = 2.
    alpha = .25
    pt_1 = tf.where(tf.equal(y_true, 1), y_pred, tf.ones_like(y_pred))
    pt_0 = tf.where(tf.equal(y_true, 0), y_pred, tf.zeros_like(y_pred))
    return -K.mean(alpha * K.pow(1. - pt_1, gamma) * K.log(pt_1)) \
            -K.mean((1 - alpha) * K.pow(pt_0, gamma) * K.log(1. - pt_0))

metrics = ["accuracy", get_f1]

model = attention_unet.att_unet(256, 256, n_label=1)

model.compile(
    optimizer=optimizer,
    loss=dice_loss,
    metrics=metrics
)

In [26]:
model.fit(
    trainDataset, 
    callbacks=[DisplayCallback(trainDataset, model), mcp_save],
    epochs=50
)

Epoch 1/50
      6/Unknown - 4s 216ms/step - loss: 0.9841 - accuracy: 0.0970 - get_f1: 5.3076e-04WARNING:tensorflow:Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0867s vs `on_train_batch_end` time: 0.1303s). Check your callbacks.
    163/Unknown - 38s 217ms/step - loss: 0.8282 - accuracy: 0.5820 - get_f1: 1.9537e-05dict_keys(['loss', 'accuracy', 'get_f1'])
New best weights found!

Epoch 1: saving model to ./attention-unet-weights/dice\weights.01-0.8282.hdf5
163/163 [==============================] - 38s 218ms/step - loss: 0.8282 - accuracy: 0.5820 - get_f1: 1.9537e-05
Epoch 2/50
163/163 [==============================] - ETA: 0s - loss: 0.6607 - accuracy: 0.8557 - get_f1: 0.0000e+00dict_keys(['loss', 'accuracy', 'get_f1'])

Epoch 2: saving model to ./attention-unet-weights/dice\weights.02-0.6607.hdf5
163/163 [==============================] - 35s 209ms/step - loss: 0.6607 - accuracy: 0.8557 - get_f1: 0.0000e+00
Epoch 3/50
163/163 [==============

In [27]:
import pandas as pd
history = model.history.history
history_df = pd.DataFrame(history)
history_df.to_csv('attention_unet_dice_history.csv', index=True)

In [28]:
TESTING_IMAGE_PATH = 'testing/images'
TESTING_MASK_PATH = 'testing/masks'

testingImagePaths = file.getSortedFilePaths(TESTING_IMAGE_PATH)
testingMaskPaths = file.getSortedFilePaths(TESTING_MASK_PATH)

testDataset = file.datasetGenerator(testingImagePaths, testingMaskPaths)
testDataset = testDataset.prefetch(buffer_size=tf.data.AUTOTUNE)

def channels_first(image, mask):
    image = tf.transpose(image, perm=[0, 3, 1, 2])
    mask = tf.transpose(mask, perm=[0, 3, 1, 2])
    return image, mask

testDataset = testDataset.map(channels_first)

testDataset

<MapDataset element_spec=(TensorSpec(shape=(None, 3, 256, 256), dtype=tf.int32, name=None), TensorSpec(shape=(None, 1, 256, 256), dtype=tf.int32, name=None))>

In [30]:
# Test dataset take 10
# Train dataset take 
predicted_masks = [];
true_masks = [];

# model.load_weights("./unet-weights/focal/best_weights.hdf5");

def displayPrediction(display_list, extra_title=''):
    plt.figure(figsize=(15, 15))
    title = ['Tinklaines nuotrauka', 'Anotacija', 'Segmentuotas vaizdas']

    if len(display_list) > len(title):
        title.append(extra_title)

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(display_list[i], cmap='gray')
        plt.axis('off')
        
    plt.show()
    
def create_mask(pred_mask):
    return pred_mask[0]
    
for image, mask in testDataset.take(55):
    pred_mask = model.predict(image)
    pred_mask = create_mask(pred_mask)
    if pred_mask.max() != 0:
        true_masks.append(mask[0])
        predicted_masks.append(pred_mask)
        # displayPrediction([image[0], mask[0], pred_mask])

In [32]:
with tf.device("cpu:0"):
    m = tf.keras.metrics.MeanIoU(num_classes=2)
    m.update_state(predicted_masks, true_masks)
    m.result().numpy()

m.result().numpy()

0.0